In [1]:
!apt-get install -y -qq nmap
!pip install streamlit plotly -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package libpcap0.8:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../0-libpcap0.8_1.10.1-4ubuntu1.22.04.1_amd64.deb ...
Unpacking libpcap0.8:amd64 (1.10.1-4ubuntu1.22.04.1) ...
Selecting previously unselected package liblinear4:amd64.
Preparing to unpack .../1-liblinear4_2.3.0+dfsg-5_amd64.deb ...
Unpacking liblinear4:amd64 (2.3.0+dfsg-5) ...
Selecting previously unselected package liblua5.3-0:amd64.
Preparing to unpack .../2-liblua5.3-0_5.3.6-1build1_amd64.deb ...
Unpacking liblua5.3-0:amd64 (5.3.6-1build1) ...
Selecting previously unselected package lua-lpeg:amd64.
Preparing to unpack .../3-lua-lpeg_1.0.2-1_amd64.deb ...
Unpacking lua-lpeg:amd64 (1.0.2-1) ...
Selecting previously unselected package nmap-common.
Preparing to unpack .../4-nmap-common_7.91+dfsg1+really7.80+dfsg1-2ubuntu0.1_all.deb ...
Unpacking nmap-common (7.91+dfsg1+really7.80+dfsg1-2ubuntu0.1) ...
Selecting previously unselected pac

In [2]:
%%writefile dashboard.py
import streamlit as st
import subprocess
import xml.etree.ElementTree as ET
import requests
import pandas as pd
import plotly.express as px
import time
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime

VT_API_KEY = os.environ.get("API_KEY")
sender_email = os.environ.get("GMAIL_SENDER")
app_password = os.environ.get("GMAIL_PASSWORD")


st.set_page_config(page_title="Threat Scanner", page_icon="🏴‍☠️", layout="wide")

st.markdown("""
    <style>
    .stApp { background-color: #0d1117; color: #c9d1d9; }
    h1, h2, h3 { color: #FFD700 !important; font-family: 'Courier New', Courier, monospace; }
    .stButton>button { background-color: #FF2400; color: white; border: 2px solid #FFD700; border-radius: 8px; font-weight: bold;}
    .stButton>button:hover { background-color: #8B0000; color: #FFD700; border: 2px solid white; }
    .stAlert { background-color: #161b22; border-left: 5px solid #005A9C; }
    hr { border-color: #FF2400; }
    </style>
""", unsafe_allow_html=True)

st.title("🏴‍☠️ Threat Scanner")
st.caption("Network health & Threat Analysis Dashboard")
st.divider()

st.sidebar.title("⚙️ Settings")
st.sidebar.caption("Configure your scanner.")

target_input = st.sidebar.text_area("Enter Targets (comma or newline separated)", value="")
targets = [t.strip() for t in target_input.replace('\n', ',').split(',') if t.strip()]

st.sidebar.divider()
st.sidebar.subheader("✉️ Email Setup")
recipient_email = st.sidebar.text_input("Alert Recipient Address")

st.sidebar.divider()
st.sidebar.subheader("📊 Status")
st.sidebar.write(f"**Targets Ready:** {len(targets)}")

# Status indicators for secrets
if VT_API_KEY:
    st.sidebar.success("VT API Key: Loaded")
else:
    st.sidebar.warning("VT API Key: Missing 🔴")

if sender_email and app_password:
    st.sidebar.success("Sender Credentials: Loaded")
else:
    st.sidebar.error("Sender Credentials: Missing 🔴")

if sender_email and app_password and recipient_email:
    st.sidebar.success("Alert System: Configured")
else:
    st.sidebar.warning("Alert System: Yet to be Configured 🔴")

run_scan = st.sidebar.button("🔍 Run Scan", use_container_width=True)


SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

def run_nmap_scan(target):
    xml_file = f"{SCAN_DIR}/{target}.xml"
    try:
        subprocess.run(["nmap", "-Pn", "-sV", "-oX", xml_file, target], capture_output=True, timeout=120)
    except Exception as e:
        pass
    return xml_file

def parse_nmap(xml_file, target):
    findings = []
    if not os.path.exists(xml_file):
        return [{"Target": target, "Vulnerability": "Host Unreachable / Invalid", "Severity": "Informational", "Score": 0, "Recommendation": "Verify target URL."}]

    try:
        root = ET.parse(xml_file).getroot()
        for host in root.findall("host"):
            for port in host.findall(".//port"):
                svc = port.find("service")
                service_name = svc.get("name") if svc is not None else "unknown"
                port_id = port.get("portid")

                if port_id in ["21", "23"] or service_name in ["ftp", "telnet"]:
                    findings.append({"Target": target, "Vulnerability": f"Cleartext Protocol Exposed ({service_name.upper()})", "Severity": "Critical", "Score": 9, "Recommendation": "Disable service; use secure alternatives like SFTP or SSH."})
                elif port_id in ["22", "3389"] or service_name in ["ssh", "ms-wbt-server"]:
                    findings.append({"Target": target, "Vulnerability": f"Remote Admin Interface Exposed ({service_name.upper()})", "Severity": "High", "Score": 7, "Recommendation": "Restrict access via firewall to trusted IPs or VPN only."})
                elif port_id in ["3306", "5432", "1433", "27017"]:
                    findings.append({"Target": target, "Vulnerability": f"Database Exposed to Public (Port {port_id})", "Severity": "High", "Score": 8, "Recommendation": "Bind database to localhost and block external port access."})
                elif port_id in ["80", "8080"] or service_name in ["http", "http-proxy"]:
                    findings.append({"Target": target, "Vulnerability": f"Unencrypted Web Traffic (Port {port_id})", "Severity": "Medium", "Score": 4, "Recommendation": "Enforce HTTPS/TLS and redirect port 80 traffic."})
                else:
                    findings.append({"Target": target, "Vulnerability": f"Open Port {port_id} ({service_name})", "Severity": "Low", "Score": 2, "Recommendation": "Ensure service is required, patched, and securely configured."})
    except Exception:
        pass
    return findings

def check_virustotal(target, api_key):
    if not api_key: return []
    try:
        r = requests.get(f"https://www.virustotal.com/api/v3/domains/{target}", headers={"x-apikey": api_key}, timeout=5)
        stats = r.json().get("data", {}).get("attributes", {}).get("last_analysis_stats", {})
        malicious = stats.get("malicious", 0)
        if malicious > 0:
            return [{"Target": target, "Vulnerability": f"Malicious Reputation ({malicious} VT engines flagged)", "Severity": "Critical", "Score": 10, "Recommendation": "Investigate immediately for compromise and rotate IPs/Domains."}]
    except Exception:
        pass
    return []


def send_automated_alert(high_crit_df, target_summary, timestamp):
    if not sender_email or not app_password or not recipient_email:
        return

    overall_risk = high_crit_df["Score"].max()
    subject_severity = "CRITICAL" if overall_risk >= 9 else "HIGH"
    targets_str = ", ".join(high_crit_df["Target"].unique())

    html = f"""
    <html>
      <body style="font-family: Arial, sans-serif; color: #333;">
        <h2 style="color: #FF2400;">🚨Threat Scanner - Security Alert</h2>
        <p><strong>Scan Timestamp:</strong> {timestamp}</p>
        <p><strong>Target(s):</strong> {targets_str}</p>
        <p><strong>Overall Risk Score:</strong> <span style="color: red; font-size: 1.2em;">{overall_risk}/10</span></p>

        <h3>High & Critical Findings Summary</h3>
        <table border="1" cellpadding="8" style="border-collapse: collapse; width: 100%;">
          <tr style="background-color: #f2f2f2;">
            <th>Target</th><th>Vulnerability</th><th>Severity</th><th>Score</th><th>Recommended Action</th>
          </tr>
    """
    for _, row in high_crit_df.iterrows():
        html += f"""
          <tr>
            <td>{row['Target']}</td><td>{row['Vulnerability']}</td>
            <td style="color: {'red' if row['Severity']=='Critical' else 'orange'};"><b>{row['Severity']}</b></td>
            <td>{row['Score']}</td><td>{row['Recommendation']}</td>
          </tr>
        """
    html += """
        </table>
        <br>
        <hr>
        <p style="font-size: 0.8em; color: #777;">
          <em>Automated Disclaimer: This email was generated automatically by the Threat Scanner system.
          Do not reply directly to this message. Information contained herein is for authorized defensive purposes only.</em>
        </p>
      </body>
    </html>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"[{subject_severity}] Vulnerability Alert for {targets_str}"
    msg["From"] = sender_email
    msg["To"] = recipient_email
    msg.attach(MIMEText(html, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender_email, app_password)
        server.send_message(msg)
        server.quit()
        st.toast("📧 Automated alert sent successfully!")
    except Exception as e:
        st.error(f"Failed to send email alert: {e}")

if "results_df" not in st.session_state:
    st.session_state.results_df = pd.DataFrame()

if run_scan:
    if not targets:
        st.error("❌ Please enter at least one target in the sidebar.")
    else:
        all_findings = []
        progress_text = st.empty()
        bar = st.progress(0)

        for i, target in enumerate(targets):
            progress_text.text(f"Scanning {target}... (Ports, Services, Reputation)")

            nmap_xml = run_nmap_scan(target)
            all_findings.extend(parse_nmap(nmap_xml, target))
            all_findings.extend(check_virustotal(target, VT_API_KEY))

            bar.progress((i + 1) / len(targets))

        st.session_state.results_df = pd.DataFrame(all_findings)
        st.session_state.scan_time = datetime.now().strftime("%d %b %Y %H:%M:%S")
        progress_text.empty()
        bar.empty()

        df = st.session_state.results_df
        if not df.empty:
            alert_df = df[df["Severity"].isin(["High", "Critical"])]
            if not alert_df.empty:
                send_automated_alert(alert_df, targets, st.session_state.scan_time)


df = st.session_state.results_df

if df.empty:
    st.info("It looks calm here. Enter targets and run the scan to see results.")
else:
    st.success(f"Scan Completed: {st.session_state.scan_time}")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("🎯Targets Scanned", df["Target"].nunique())
    c2.metric("⛓️‍💥Total Vulnerabilities", len(df))
    c3.metric("⚠️Critical/High Threats", len(df[df["Severity"].isin(["Critical", "High"])]))
    c4.metric("🏴‍☠️Max Risk Score", df["Score"].max())
    st.divider()

    st.subheader("📜 Summary of the Scan")

    def color_severity(val):
        colors = {'Critical': 'color: #FF0000; font-weight: bold;', 'High': 'color: #FF4500;',
                  'Medium': 'color: #FFD700;', 'Low': 'color: #1E90FF;', 'Informational': 'color: #A9A9A9;'}
        return colors.get(val, '')

    styled_df = df.style.map(color_severity, subset=['Severity'])
    st.dataframe(styled_df, use_container_width=True, hide_index=True)
    st.divider()

    st.subheader("📈 Observed Trends")
    ch1, ch2 = st.columns(2)

    with ch1:
        sev_counts = df["Severity"].value_counts().reset_index()
        sev_counts.columns = ["Severity", "Count"]
        fig_pie = px.pie(sev_counts, names="Severity", values="Count", hole=0.4,
                         title="Overall Vulnerability Distribution",
                         color="Severity",
                         color_discrete_map={"Critical": "#8B0000", "High": "#FF2400", "Medium": "#FFD700", "Low": "#005A9C", "Informational": "#808080"})
        fig_pie.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white")
        st.plotly_chart(fig_pie, use_container_width=True)

    with ch2:
        target_counts = df.groupby(["Target", "Severity"]).size().reset_index(name="Count")
        fig_bar = px.bar(target_counts, x="Target", y="Count", color="Severity",
                         title="Volume of Vulnerabilities by Target",
                         color_discrete_map={"Critical": "#8B0000", "High": "#FF2400", "Medium": "#FFD700", "Low": "#005A9C", "Informational": "#808080"})
        fig_bar.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white")
        st.plotly_chart(fig_bar, use_container_width=True)

    ch3, ch4 = st.columns(2)

    with ch3:
        max_score_df = df.groupby("Target")["Score"].max().reset_index()
        fig_line = px.line(max_score_df, x="Target", y="Score", markers=True,
                           title="Target vs Score",
                           color_discrete_sequence=["#FFD700"])

        fig_line.update_traces(marker=dict(size=12, color="#FF2400", line=dict(width=2, color="white")))
        fig_line.update_layout(
            paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white",
            yaxis=dict(range=[0, 11], title="Max Risk Score"),
            xaxis=dict(title="Target")
        )
        st.plotly_chart(fig_line, use_container_width=True)

    with ch4:
        vuln_counts = df["Vulnerability"].value_counts().reset_index()
        vuln_counts.columns = ["Vulnerability", "Count"]

        fig_hbar = px.bar(vuln_counts.head(5), x="Count", y="Vulnerability", orientation='h',
                          title="Top 5 Frequent Issues",
                          color="Count", color_continuous_scale="Reds")

        fig_hbar.update_layout(
            paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white",
            yaxis={'categoryorder':'total ascending'}
        )
        st.plotly_chart(fig_hbar, use_container_width=True)

    st.divider()

    st.subheader("⚠️ Top Vulnerabilities")
    top_vulns = df[df["Severity"].isin(["Critical", "High", "Medium"])].sort_values(by="Score", ascending=False)
    if top_vulns.empty:
        st.write("No severe threats found. Your ship is safe!")
    else:
        for idx, row in top_vulns.head(5).iterrows():
            st.error(f"**{row['Target']}** - {row['Vulnerability']} (Score: {row['Score']})  \n*Recommendation:* {row['Recommendation']}")

Writing dashboard.py


In [3]:
import os
lines = len(open('dashboard.py', encoding='utf-8').readlines())
size  = os.path.getsize('dashboard.py')
print(f'dashboard.py ✅ {lines} lines  |  {size:,} bytes')

os.makedirs('.streamlit', exist_ok=True)
config = '''
[theme]
primaryColor             = "#FF2400"
backgroundColor          = "#0d1117"
secondaryBackgroundColor = "#161b22"
textColor                = "#c9d1d9"
font                     = "monospace"
'''
open('.streamlit/config.toml', 'w').write(config)
print('.streamlit/config.toml  ✅')
print()
print('Now run the two launch cells below ↓')

dashboard.py ✅ 294 lines  |  12,980 bytes
.streamlit/config.toml  ✅

Now run the two launch cells below ↓


In [4]:
import subprocess, threading, time
import os
from google.colab import userdata

def run_streamlit():
    api_key = userdata.get("API_KEY")
    gmail_sender = userdata.get("GMAIL_SENDER")
    gmail_password = userdata.get("GMAIL_PASSWORD")

    env_vars = os.environ.copy()
    if api_key: env_vars["API_KEY"] = api_key
    if gmail_sender: env_vars["GMAIL_SENDER"] = gmail_sender
    if gmail_password: env_vars["GMAIL_PASSWORD"] = gmail_password

    subprocess.run(['streamlit', 'run', 'dashboard.py',
                    '--server.port', '8501', '--server.headless', 'true'], env=env_vars)

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(5)
print('Streamlit running on port 8501')
print('Now run Cell B below to open the Cloudflare tunnel.')

Streamlit running on port 8501
Now run Cell B below to open the Cloudflare tunnel.


In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-03-16T10:44:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-16T10:44:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-16T10:44:54Z INF +--------------------------------------------------------------------------------------------+
2026-03-16T10:44:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-16T10:44:54Z INF |  https://gratuit-refine-venue-object.trycloudflare.com